# Methane ASL properties

**Evidence: numerical verification.** Evaluate the compiled pressure-property callback at 298.15 K and 100,000 Pa, methane vapor.


Gross and Sadowski (2001), Table 2
([paper](https://doi.org/10.1021/ie0003887)), supplies the methane coefficients
in `gross-2001-methane-ethane/3`. McBride, Zehe and Gordon (2002),
[NASA/TP-2002-211556](https://ntrs.nasa.gov/citations/20020085330), Appendix D,
supplies the NASA9 methane record in `nasa-glenn-2002-hda-species/1`.
The packet preserves the source gas constant, 8.314510 J/(mol K), and
100,000 Pa standard pressure; `ThermochemistryPacket` performs the native
gas-constant conversion. The first methane interval is 200–1000 K.
Zotero parent: `AQDUURW4`; PDF: `SYJLCAXZ`.


## Read the inputs

Use a repository checkout with the environment described in the installation
guide. The wheel alone does not contain these data. All density, association,
equilibrium and parameter-fit iterations below run in the native Engine.


In [1]:
from pathlib import Path
import json
import math
import csv
import pandas as pd
from IPython.display import display

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "ARCHITECTURE.yaml").is_file()
    )
)
PACKETS = ROOT / "data" / "packets"
from epcsaft import (
    Parameters,
    Pure,
    Mixture,
    ThermochemistryPacket,
    ThermochemistryStatus,
    GAS_CONSTANT_J_PER_MOL_K,
    PropertyObservable,
    PressureActionDirection,
    PropertyActionStatus,
)
from epcsaft import equilibrium as eq, regression as rg
from epcsaft import _core
from epcsaft.eos import property_asl_record


## Load the physical records

An ASL record serializes the installed model for an external function.
Locate `libepcsaft_asl.so` beside the installed `_core` extension.
This example needs Pyomo and IDAES, but no external Ipopt executable.


In [2]:
p = Parameters.from_json(
    PACKETS / "gross-2001-methane-ethane/3/parameters/parameters.json",
    components=["methane"],
)
thermal = ThermochemistryPacket(
    ROOT / "data/packets/nasa-glenn-2002-hda-species/1/parameters/nasa9-records.json"
).select(["methane"])
model = Pure(p, thermochemistry=thermal)


## Compare callback values and exact derivatives

Rows are molar density, total enthalpy and ln fugacity coefficient. Numerical
arguments are `(T / K, P / Pa, w / 1)` with pure-component weight w = 1.
Compare values, T/P gradients and TT/TP/PP Hessian entries with native actions
normalized by 1 K and 1 Pa increments. The criterion is 10⁻¹⁰ relative with
a 10⁻²² absolute floor. Hessians are packed column-wise as TT, TP, PP, Tw,
Pw, ww. Gradient units are observable-unit per K or Pa; second derivatives
divide by the two corresponding units. The complete arrays are printed.


In [3]:
from pyomo.environ import ExternalFunction
import idaes

text = property_asl_record(model)
fn = ExternalFunction(
    library=str(Path(_core.__file__).resolve().with_name("libepcsaft_asl.so")),
    function="epcsaft_pressure_property",
)
t, p = (PressureActionDirection(temperature=1), PressureActionDirection(pressure=1))
obs = [PropertyObservable.TotalEnthalpy, PropertyObservable.LogFugacityCoefficient]
batches = [
    model.pressure_actions(
        298.15, 100000.0, phase="vapor", observables=obs, directions=ds
    )
    for ds in ([t], [p], [t, t], [t, p], [p, p])
]
native = [
    [
        batches[0].molar_density,
        *(b.density_action[0] for b in batches[:2]),
        *(b.density_mixed_action for b in batches[2:]),
    ]
]
native += [
    [batches[0].results[k].value, *(b.results[k].action for b in batches)]
    for k in range(2)
]
rows = []
for row, expected in enumerate(native):
    value, gradient, hessian = fn.evaluate_fgh(
        (298.15, 100000.0, 1.0, text, str(row), "1")
    )
    actual = [value, *gradient[:2], *hessian[:3]]
    assert all(
        (abs(a - b) <= 1e-10 * max(abs(b), 1e-12) for a, b in zip(actual, expected))
    )
    rows.append(
        dict(
            value=value,
            gradient=gradient[:3],
            hessian=hessian[:6],
            maximum_absolute_error=max((abs(a - b) for a, b in zip(actual, expected))),
        )
    )
values = dict(rows=rows, idaes_version=idaes.__version__, record_characters=len(text))
display(
    pd.DataFrame(
        {
            "Quantity": [
                "Density / mol m^-3",
                "Total enthalpy / J mol^-1",
                "ln fugacity coefficient / 1",
            ],
            "Value": [row["value"] for row in rows],
            "Maximum absolute transport error": [
                row["maximum_absolute_error"] for row in rows
            ],
        }
    )
)


epcsaft_property prepared record 1 of at most 16


,Quantity,Value,Maximum absolute transport error
0,Density / mol m^-3,40.415468,0.0
1,Total enthalpy / J mol^-1,-74616.276249,0.0
2,ln fugacity coefficient / 1,-0.001879,0.0


## Meaning and limits


The comparison verifies value and derivative transport for this neutral
pressure chart. It neither solves a flowsheet nor qualifies ionic or reacting
process derivatives. See the two-liquid settler example for a live IDAES outer
optimization. Thermodynamic accuracy retains the limits of the underlying
parameter and ideal thermal records.


In [4]:
print(json.dumps(values, indent=2))


{
  "rows": [
    {
      "value": 40.41546780540215,
      "gradient": [
        -0.13644643130551143,
        0.0004049148313984332,
        0.0
      ],
      "hessian": [
        0.0009264549579580186,
        -1.3759822413945478e-06,
        1.5221669385235544e-11,
        -0.0,
        -0.0,
        -0.0
      ],
      "maximum_absolute_error": 0.0
    },
    {
      "value": -74616.2762493091,
      "gradient": [
        35.784845922598294,
        -0.00016287134409514559,
        0.0
      ],
      "hessian": [
        0.036339805490723215,
        9.387477215379765e-07,
        -2.174992797018837e-12,
        0.0,
        0.0,
        0.0
      ],
      "maximum_absolute_error": 0.0
    },
    {
      "value": -0.001879158636526126,
      "gradient": [
        2.2021702489643104e-05,
        -1.8785446370241538e-08,
        0.0
      ],
      "hessian": [
        -2.7425433027737104e-07,
        2.2036411212874154e-10,
        1.230322424039763e-16,
        0.0,
        0.0,
 